# 金融计量经济学数据分析
- 使用 final_data.csv 进行数据分析
- 简介: 通过Python对CSV数据进行一系列数据分析和可视化

## 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置图形样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("库导入成功！")

## 配置参数

In [ ]:
# 分析配置参数
MAX_SCATTER_MATRIX_VARS = 5  # 散点图矩阵最多显示的变量数量
HISTOGRAM_BINS = 30  # 直方图的分箱数量
FIGURE_DPI = 300  # 图形保存的DPI

print(f"配置参数已设置:")
print(f"  - 散点图矩阵最大变量数: {MAX_SCATTER_MATRIX_VARS}")
print(f"  - 直方图分箱数: {HISTOGRAM_BINS}")
print(f"  - 图形DPI: {FIGURE_DPI}")

## 加载数据
从 final_data.csv 文件中加载数据

In [ ]:
# 读取CSV文件
try:
    df = pd.read_csv('final_data.csv')
    print(f"数据加载成功！")
    print(f"数据形状: {df.shape}")
    print(f"\n列名: {list(df.columns)}")
except FileNotFoundError:
    print("错误: 未找到 final_data.csv 文件")
    print("请确保 final_data.csv 文件位于当前目录下")
    df = None

## 查看数据前几行

In [ ]:
if df is not None:
    print("数据的前5行:")
    display(df.head())
    
    print("\n数据的后5行:")
    display(df.tail())

## 数据基本信息

In [ ]:
if df is not None:
    print("数据类型和缺失值信息:")
    print(df.info())
    
    print("\n缺失值统计:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("没有缺失值")

## 描述性统计
计算数值变量的均值、标准差、最小值、最大值、中位数等统计量

In [ ]:
if df is not None:
    # 获取数值型列
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols) > 0:
        print("数值变量的描述性统计:")
        desc_stats = df[numeric_cols].describe()
        display(desc_stats)
        
        # 添加额外的统计量
        print("\n额外统计量 (偏度和峰度):")
        extra_stats = pd.DataFrame({
            '偏度': df[numeric_cols].skew(),
            '峰度': df[numeric_cols].kurtosis()
        })
        display(extra_stats)
        
        # 保存统计结果到CSV
        desc_stats.to_csv('descriptive_statistics.csv')
        print("\n描述性统计已保存到 descriptive_statistics.csv")
    else:
        print("数据中没有数值型变量")

## 数据分布可视化 - 直方图
绘制所有数值变量的直方图

In [ ]:
if df is not None and len(numeric_cols) > 0:
    # 计算子图布局
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numeric_cols) > 1 else [axes]
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        df[col].hist(bins=HISTOGRAM_BINS, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(f'{col} 的直方图', fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('频数')
        ax.grid(True, alpha=0.3)
    
    # 隐藏多余的子图
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('histograms.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("直方图已保存为 histograms.png")

## 密度函数图 (核密度估计)
绘制变量的核密度估计图

In [ ]:
if df is not None and len(numeric_cols) > 0:
    plt.figure(figsize=(12, 6))
    
    for col in numeric_cols:
        df[col].plot(kind='density', label=col, linewidth=2)
    
    plt.title('变量的核密度估计图', fontsize=14, fontweight='bold')
    plt.xlabel('值')
    plt.ylabel('密度')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('density_plots.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("密度图已保存为 density_plots.png")

## 箱线图
绘制箱线图以显示数据的分布和异常值

In [ ]:
if df is not None and len(numeric_cols) > 0:
    plt.figure(figsize=(12, 6))
    
    # 创建箱线图
    df[numeric_cols].boxplot()
    plt.title('变量的箱线图', fontsize=14, fontweight='bold')
    plt.ylabel('值')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('box_plots.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("箱线图已保存为 box_plots.png")

## 小提琴图
绘制小提琴图以同时显示数据的分布和密度

In [ ]:
if df is not None and len(numeric_cols) > 0:
    # 准备数据用于小提琴图
    df_melted = df[numeric_cols].melt(var_name='变量', value_name='值')
    
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=df_melted, x='变量', y='值', palette='Set2')
    plt.title('变量的小提琴图', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('violin_plots.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("小提琴图已保存为 violin_plots.png")

## 相关性分析
计算变量之间的相关系数

In [ ]:
if df is not None and len(numeric_cols) > 0:
    # 计算相关系数矩阵
    correlation_matrix = df[numeric_cols].corr()
    
    print("相关系数矩阵:")
    display(correlation_matrix)
    
    # 绘制相关性热力图
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8},
                fmt='.3f')
    plt.title('变量相关性热力图', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("相关性热力图已保存为 correlation_heatmap.png")
    
    # 保存相关系数矩阵
    correlation_matrix.to_csv('correlation_matrix.csv')
    print("相关系数矩阵已保存为 correlation_matrix.csv")

## 散点图矩阵
绘制变量之间的散点图矩阵以观察两两关系

In [ ]:
if df is not None and len(numeric_cols) > 0:
    # 如果变量太多，只选择前几个
    cols_to_plot = numeric_cols[:min(MAX_SCATTER_MATRIX_VARS, len(numeric_cols))]
    
    if len(cols_to_plot) >= 2:
        # 创建散点图矩阵
        pairplot = sns.pairplot(df[cols_to_plot], diag_kind='kde', 
                                plot_kws={'alpha': 0.6, 's': 30},
                                diag_kws={'linewidth': 2})
        pairplot.fig.suptitle('变量散点图矩阵', y=1.02, fontsize=14, fontweight='bold')
        plt.savefig('scatter_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("散点图矩阵已保存为 scatter_matrix.png")
    else:
        print("数值变量少于2个，无法绘制散点图矩阵")

## 单变量散点图示例
如果数据中有至少两个变量，绘制第一个变量和第二个变量的散点图

In [ ]:
if df is not None and len(numeric_cols) >= 2:
    var1 = numeric_cols[0]
    var2 = numeric_cols[1]
    
    plt.figure(figsize=(10, 6))
    plt.scatter(df[var1], df[var2], alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    # 添加拟合线
    z = np.polyfit(df[var1].dropna(), df[var2].dropna(), 1)
    p = np.poly1d(z)
    plt.plot(df[var1].sort_values(), p(df[var1].sort_values()), 
             "r-", linewidth=2, label='拟合线')
    
    # 计算相关系数
    corr_coef = df[var1].corr(df[var2])
    
    plt.title(f'{var1} vs {var2} 散点图\n相关系数: {corr_coef:.4f}', 
              fontsize=14, fontweight='bold')
    plt.xlabel(var1, fontsize=12)
    plt.ylabel(var2, fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('scatter_plot_example.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("散点图已保存为 scatter_plot_example.png")

## 数据摘要报告
生成完整的数据分析报告

In [ ]:
if df is not None:
    print("="*60)
    print("数据分析摘要报告")
    print("="*60)
    print(f"\n1. 数据基本信息:")
    print(f"   - 样本数量: {len(df)}")
    print(f"   - 变量数量: {len(df.columns)}")
    print(f"   - 数值变量数量: {len(numeric_cols)}")
    print(f"   - 分类变量数量: {len(df.select_dtypes(include=['object']).columns)}")
    
    print(f"\n2. 缺失值情况:")
    total_missing = df.isnull().sum().sum()
    if total_missing > 0:
        print(f"   - 总缺失值数量: {total_missing}")
        print(f"   - 缺失值比例: {total_missing / (len(df) * len(df.columns)) * 100:.2f}%")
    else:
        print(f"   - 无缺失值")
    
    if len(numeric_cols) > 0:
        print(f"\n3. 数值变量统计:")
        for col in numeric_cols:
            print(f"\n   {col}:")
            print(f"     - 均值: {df[col].mean():.4f}")
            print(f"     - 标准差: {df[col].std():.4f}")
            print(f"     - 最小值: {df[col].min():.4f}")
            print(f"     - 最大值: {df[col].max():.4f}")
            print(f"     - 中位数: {df[col].median():.4f}")
    
    print(f"\n4. 生成的文件:")
    output_files = [
        'descriptive_statistics.csv',
        'correlation_matrix.csv',
        'histograms.png',
        'density_plots.png',
        'box_plots.png',
        'violin_plots.png',
        'correlation_heatmap.png',
        'scatter_matrix.png',
        'scatter_plot_example.png'
    ]
    for file in output_files:
        print(f"   - {file}")
    
    print("\n" + "="*60)
    print("分析完成！")
    print("="*60)